<a href="https://colab.research.google.com/github/MirMurtazaa/agent-miscon/blob/main/simulation_animation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import networkx as nx
import json
import random
import os
from google.colab import files
from tqdm import tqdm

# 1. Strategy Mapping (matching paper heuristics)
strategies = {
    0: 'HUB', 1: 'BRIDGE', 2: 'CLOSENESS',
    3: 'K_CORE', 4: 'PERIPHERAL', 5: 'HIGH_CLUSTERING',
    6: 'OPPONENT_NEIGHBOR'
}

exp_files = {
    'EXP1: Naive vs Naive (IND-IND)': 'e1_evaluation_strategy_log.csv',
    'EXP2: Rational Blocker (MM-IND)': 'e2_evaluation_strategy_log.csv',
    'EXP3: Rational vs Rational (MM-MM)': 'e3_evaluation_strategy_log.csv',
    'EXP4: Rational Adversary (IND-MM)': 'e4_evaluation_strategy_log.csv'
}

# 2. Exact Strategy Selector
def select_exact_node(strategy_name, free_nodes, opponent_seeds, G, metrics):
    if not free_nodes:
        return None

    if strategy_name == 'OPPONENT_NEIGHBOR':
        scores = {n: sum(1 for neighbor in G.neighbors(n) if neighbor in opponent_seeds) for n in free_nodes}
    elif strategy_name == 'PERIPHERAL':
        scores = {n: -metrics['degree'][n] for n in free_nodes}
    else:
        metric_map = {
            'HUB': 'degree',
            'BRIDGE': 'betweenness',
            'CLOSENESS': 'closeness',
            'K_CORE': 'core',
            'HIGH_CLUSTERING': 'clustering'
        }
        m_name = metric_map.get(strategy_name, 'degree')
        scores = {n: metrics[m_name][n] for n in free_nodes}

    max_score = max(scores.values())
    candidates = [n for n, v in scores.items() if v == max_score]
    return random.choice(candidates)

# 3. Load all experiment data
dfs = {}
for exp_name, file_name in exp_files.items():
    if os.path.exists(file_name):
        dfs[exp_name] = pd.read_csv(file_name)
    else:
        print(f"Warning: {file_name} not found. Please upload it.")

if not dfs:
    raise ValueError("No evaluation logs found. Please upload the CSV files.")

first_df = list(dfs.values())[0]

# Extract exactly 3 networks: one BA, one ER, one WS
target_networks = []
for n in first_df['network'].unique():
    if 'BA' in n and not any('BA' in tn for tn in target_networks): target_networks.append(n)
    if 'ER' in n and not any('ER' in tn for tn in target_networks): target_networks.append(n)
    if 'WS' in n and not any('WS' in tn for tn in target_networks): target_networks.append(n)
target_networks = target_networks[:3]

networks_data = []

# 4. Generate combined simulation traces
print("Preparing graph structures and computing metrics...")
for net_name in tqdm(target_networks, desc="Processing Networks"):
    N = 800
    if 'BA' in net_name:
        G = nx.barabasi_albert_graph(N, 3, seed=42)
    elif 'ER' in net_name:
        G = nx.erdos_renyi_graph(N, 0.01, seed=42)
    else:
        G = nx.watts_strogatz_graph(N, 8, 0.1, seed=42)

    pos = nx.spring_layout(G, seed=42, iterations=50)

    metrics = {
        'degree': dict(G.degree()),
        'betweenness': nx.betweenness_centrality(G),
        'closeness': nx.closeness_centrality(G),
        'core': nx.core_number(G),
        'clustering': nx.clustering(G)
    }

    node_idx = {n: i for i, n in enumerate(G.nodes())}
    nodes = [{
        "id": i, "degree": metrics['degree'][n], "x": float(pos[n][0]), "y": float(pos[n][1])
    } for n, i in node_idx.items()]

    links = [{"source": node_idx[u], "target": node_idx[v]} for u, v in G.edges()]

    experiments_data = {}

    for exp_name, df in dfs.items():
        game_df = df[(df['network'] == net_name) & (df['game_num'] == 0)].sort_values('round')
        if len(game_df) == 0: continue

        rounds_data = []
        persistent_adv = set()
        persistent_block = set()
        adv_history = {} # Track strategy for every selected adv node
        blk_history = {} # Track strategy for every selected blk node
        free_nodes = set(G.nodes())

        for _, row in game_df.iterrows():
            rnd = int(row['round']) + 1
            adv_strat = strategies.get(row['adversary_strategy_id'], 'HUB')
            blk_strat = strategies.get(row['blocker_strategy_id'], 'HUB')
            norm_inf = float(row['norm_influence'])

            new_adv = select_exact_node(adv_strat, free_nodes, persistent_block, G, metrics)
            if new_adv is not None:
                persistent_adv.add(new_adv)
                free_nodes.remove(new_adv)
                adv_history[new_adv] = adv_strat

            new_blk = select_exact_node(blk_strat, free_nodes, persistent_adv, G, metrics)
            if new_blk is not None:
                persistent_block.add(new_blk)
                free_nodes.remove(new_blk)
                blk_history[new_blk] = blk_strat

            target_total_inf = int(norm_inf * N)
            transient_needed = max(0, target_total_inf - len(persistent_adv))
            transient_nodes = set()

            if transient_needed > 0:
                active_frontier = list(persistent_adv)
                visited = set(persistent_adv) | set(persistent_block)

                while active_frontier and len(transient_nodes) < transient_needed:
                    curr = random.choice(active_frontier)
                    unvisited_neighbors = [n for n in G.neighbors(curr) if n not in visited]

                    if not unvisited_neighbors:
                        active_frontier.remove(curr)
                        continue

                    chosen = random.choice(unvisited_neighbors)
                    visited.add(chosen)
                    transient_nodes.add(chosen)
                    active_frontier.append(chosen)

            rounds_data.append({
                "round": rnd,
                "adv_strategy": adv_strat,
                "block_strategy": blk_strat,
                "adv_seeds": list(persistent_adv),
                "block_seeds": list(persistent_block),
                "adv_history": dict(adv_history),
                "blk_history": dict(blk_history),
                "new_adv": new_adv,
                "new_blk": new_blk,
                "transient_nodes": list(transient_nodes),
                "norm_influence": norm_inf,
                "abs_influence": target_total_inf
            })

        experiments_data[exp_name] = rounds_data
    networks_data.append({"network": net_name, "nodes": nodes, "links": links, "experiments": experiments_data})

json_data = json.dumps(networks_data)

# 5. Create HTML with exact text boundary box calculation & tooltip-only history
html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>AGENT-MISCON Simulator (Dark Mode)</title>
    <script src="https://d3js.org/d3.v7.min.js"></script>
    <style>
        :root {{
            --adv-color: #ff4757;
            --blk-color: #1e90ff;
            --free-color: #4b4b4b;
            --trans-color: #ffa502;
            --bg-dark: #121212;
            --panel-bg: #1e272e;
            --text-main: #f1f2f6;
            --text-muted: #a4b0be;
            --border-color: #2f3542;
        }}

        body {{ font-family: 'Segoe UI', system-ui, sans-serif; margin: 0; display: flex; height: 100vh; overflow: hidden; background: var(--bg-dark); color: var(--text-main); }}

        #loading-overlay {{
            position: fixed; top: 0; left: 0; width: 100%; height: 100%;
            background: var(--bg-dark); z-index: 9999;
            display: flex; flex-direction: column; justify-content: center; align-items: center;
            transition: opacity 0.6s ease, visibility 0.6s ease;
        }}
        .spinner {{
            width: 60px; height: 60px; border: 6px solid var(--border-color);
            border-top-color: var(--adv-color); border-bottom-color: var(--blk-color);
            border-radius: 50%; animation: spin 1.2s cubic-bezier(0.5, 0.1, 0.4, 0.9) infinite;
            margin-bottom: 20px;
        }}
        @keyframes spin {{ 100% {{ transform: rotate(360deg); }} }}
        .loading-text {{ font-size: 1.2rem; font-weight: bold; color: var(--text-main); letter-spacing: 1px; }}

        #sidebar {{ width: 420px; background: var(--panel-bg); padding: 20px; box-sizing: border-box; overflow-y: auto; border-right: 1px solid var(--border-color); box-shadow: 4px 0 20px rgba(0,0,0,0.8); display: flex; flex-direction: column; z-index: 10; }}

        #graph-container {{ flex-grow: 1; position: relative; display: flex; flex-direction: column; background: var(--bg-dark); }}

        #guide {{ position: absolute; top: 20px; left: 20px; background: rgba(30, 39, 46, 0.85); padding: 12px 18px; border-radius: 8px; box-shadow: 0 4px 15px rgba(0,0,0,0.8); font-size: 0.9rem; color: var(--text-main); z-index: 20; border-left: 4px solid var(--blk-color); pointer-events: none; border: 1px solid var(--border-color); backdrop-filter: blur(4px); }}
        #guide b {{ color: var(--blk-color); }}

        #graph {{ flex-grow: 1; cursor: grab; background: transparent; }}
        #graph:active {{ cursor: grabbing; }}

        h2 {{ font-size: 1.3rem; margin: 0 0 15px 0; color: var(--text-main); border-bottom: 3px solid var(--blk-color); padding-bottom: 10px; }}
        h3 {{ font-size: 0.9rem; margin: 10px 0 10px 0; color: var(--text-muted); text-transform: uppercase; border-bottom: 1px solid var(--border-color); padding-bottom: 5px; }}

        @keyframes pulse-adv {{
            0% {{ stroke-width: 2px !important; stroke: var(--adv-color) !important; filter: drop-shadow(0 0 5px var(--adv-color)); }}
            50% {{ stroke-width: 28px !important; stroke: var(--adv-color) !important; filter: drop-shadow(0 0 40px var(--adv-color)); opacity: 0.85; }}
            100% {{ stroke-width: 2px !important; stroke: var(--adv-color) !important; filter: drop-shadow(0 0 5px var(--adv-color)); }}
        }}
        @keyframes pulse-blk {{
            0% {{ stroke-width: 2px !important; stroke: var(--blk-color) !important; filter: drop-shadow(0 0 5px var(--blk-color)); }}
            50% {{ stroke-width: 28px !important; stroke: var(--blk-color) !important; filter: drop-shadow(0 0 40px var(--blk-color)); opacity: 0.85; }}
            100% {{ stroke-width: 2px !important; stroke: var(--blk-color) !important; filter: drop-shadow(0 0 5px var(--blk-color)); }}
        }}
        .node-new-adv {{ animation: pulse-adv 1.5s infinite ease-in-out; }}
        .node-new-blk {{ animation: pulse-blk 1.5s infinite ease-in-out; }}

        .node {{ stroke: #121212; stroke-width: 1px; transition: stroke 0.2s, fill 0.4s; cursor: pointer; }}
        .link {{ stroke: #57606f; stroke-opacity: 0.3; transition: stroke-opacity 0.2s, stroke 0.2s, stroke-width 0.2s; }}

        label {{ font-size: 0.85rem; font-weight: 600; color: var(--text-muted); margin-bottom: 5px; display: block; }}
        select {{ width: 100%; padding: 8px; margin-bottom: 12px; border: 1px solid var(--border-color); border-radius: 6px; background: #2f3542; color: var(--text-main); font-size: 14px; outline: none; transition: border-color 0.2s; }}

        .btn-group {{ display: grid; grid-template-columns: 1fr 1fr; gap: 8px; margin-bottom: 15px; }}
        button {{ padding: 10px; border: none; border-radius: 6px; background: var(--blk-color); color: white; cursor: pointer; font-weight: bold; transition: all 0.2s; }}
        button:hover:not(:disabled) {{ background: #70a1ff; transform: translateY(-1px); }}
        button:disabled {{ background: #57606f; color: #a4b0be; cursor: not-allowed; }}
        #btnReset {{ background: var(--adv-color); grid-column: span 2; }}
        #btnReset:hover:not(:disabled) {{ background: #ff6b81; }}

        .tracker-header {{ display: flex; justify-content: space-between; align-items: baseline; margin-bottom: 6px; }}
        .tracker-title {{ font-size: 0.85rem; color: var(--text-muted); text-transform: uppercase; font-weight: bold; letter-spacing: 0.5px; }}
        .tracker-pct {{ font-size: 1.5rem; font-weight: 900; color: var(--adv-color); font-family: monospace; text-shadow: 0 0 10px rgba(255, 71, 87, 0.4); }}

        .progress-container {{ width: 100%; background: #2f3542; border-radius: 6px; height: 20px; overflow: hidden; margin-bottom: 15px; border: 1px solid #121212; box-shadow: inset 0 2px 4px rgba(0,0,0,0.5); }}
        .progress-bar {{ height: 100%; background: linear-gradient(90deg, #ff4757, #ff6b81); width: 0%; transition: width 0.4s ease; box-shadow: 0 0 15px rgba(255, 71, 87, 0.6); }}

        .stat-grid {{ display: grid; grid-template-columns: 1fr 1fr; gap: 10px; margin-bottom: 10px; }}
        .stat-box {{ background: #2f3542; padding: 10px; border-radius: 6px; border: 1px solid var(--border-color); text-align: center; }}
        .stat-box.adv {{ border-bottom: 3px solid var(--adv-color); }}
        .stat-box.blk {{ border-bottom: 3px solid var(--blk-color); }}
        .stat-title {{ font-size: 0.75rem; color: var(--text-muted); font-weight: bold; display: block; margin-bottom: 5px; }}
        .stat-val {{ font-size: 1.1rem; font-weight: 800; color: var(--text-main); }}

        .legend-item {{ display: flex; align-items: center; margin-bottom: 6px; font-size: 0.85rem; color: var(--text-main); }}
        .legend-color {{ width: 14px; height: 14px; margin-right: 12px; border-radius: 50%; box-shadow: inset 0 0 0 1px rgba(255,255,255,0.1); }}

        div.tooltip {{ position: absolute; padding: 14px; font: 13px sans-serif; background: rgba(30, 39, 46, 0.95); color: #fff; border-radius: 8px; pointer-events: none; opacity: 0; z-index: 100; box-shadow: 0 6px 20px rgba(0,0,0,0.8); border: 1px solid var(--border-color); backdrop-filter: blur(4px); }}
        .tooltip-title {{ font-weight: 800; margin-bottom: 6px; border-bottom: 1px solid rgba(255,255,255,0.2); padding-bottom: 4px; font-size: 14px; color: var(--blk-color); }}

        #chart-container {{ margin-top: auto; padding-top: 15px; border-top: 1px solid var(--border-color); }}
        .line {{ fill: none; stroke: var(--adv-color); stroke-width: 2.5px; filter: drop-shadow(0 2px 4px rgba(255,71,87,0.5)); }}
        .axis text {{ font-size: 10px; fill: var(--text-muted); }}
        .axis path, .axis line {{ stroke: var(--border-color); }}
    </style>
</head>
<body>
    <div id="loading-overlay">
        <div class="spinner"></div>
        <div class="loading-text">Simulating Graph Physics...</div>
    </div>

    <div id="sidebar">
        <h2>Strategy Simulator</h2>
        <label>1. Select Network Topology</label>
        <select id="netSelect"></select>
        <label>2. Select Matchup</label>
        <select id="expSelect"></select>

        <div class="btn-group">
            <button id="btnPlay">▶ Play</button>
            <button id="btnPause" disabled>⏸ Pause</button>
            <button id="btnNext">⏭ Step</button>
            <button id="btnReset">↺ Reset</button>
        </div>

        <div class="tracker-header">
            <span class="tracker-title">Influence Tracker</span>
            <span class="tracker-pct" id="infText">0.0%</span>
        </div>
        <div class="progress-container">
            <div class="progress-bar" id="infBar"></div>
        </div>

        <div class="stat-grid">
            <div class="stat-box">
                <span class="stat-title">Round</span>
                <span class="stat-val"><span id="lblRound">0</span> / 30</span>
            </div>
            <div class="stat-box">
                <span class="stat-title">Nodes Lost</span>
                <span class="stat-val" id="lblAbsInf">0</span>
            </div>
        </div>

        <h3>Current Actions</h3>
        <div class="stat-grid">
            <div class="stat-box adv">
                <span class="stat-title" style="color:var(--adv-color)">Adversary</span>
                <span class="stat-val" id="lblAdvStrat" style="font-size:0.85rem;">-</span>
            </div>
            <div class="stat-box blk">
                <span class="stat-title" style="color:var(--blk-color)">Blocker</span>
                <span class="stat-val" id="lblBlockStrat" style="font-size:0.85rem;">-</span>
            </div>
        </div>

        <h3>Legend</h3>
        <div class="legend-item"><div class="legend-color" style="background: var(--free-color);"></div> Free Node</div>
        <div class="legend-item"><div class="legend-color" style="background: var(--adv-color);"></div> Adv Seed</div>
        <div class="legend-item"><div class="legend-color" style="background: var(--blk-color);"></div> Immunized Node</div>
        <div class="legend-item"><div class="legend-color" style="background: var(--trans-color);"></div> Transient Spread</div>
        <div class="legend-item" style="margin-top: 10px;">
            <div class="legend-color" style="background: transparent; border: 4px solid var(--adv-color); width: 8px; height: 8px;"></div>
            New Adv Action (Red Pulse)
        </div>
        <div class="legend-item">
            <div class="legend-color" style="background: transparent; border: 4px solid var(--blk-color); width: 8px; height: 8px;"></div>
            New Blocker Action (Blue Pulse)
        </div>

        <div id="chart-container">
            <svg id="lineChart" width="100%" height="90"></svg>
        </div>
    </div>

    <div id="graph-container">
        <div id="guide">
            <strong>💡 Interaction Guide:</strong><br>
            <span style="color:var(--text-muted);">• <b>Hover</b> a node to view its ID, Degree, and Status.<br>
            • <b>Hover</b> a Pulsing node to see its exact Strategy.<br>
            • <b>Click</b> a node to lock/unlock its neighbors.</span>
        </div>
        <div id="graph"><svg width="100%" height="100%"></svg></div>
    </div>

    <div id="tooltip" class="tooltip"></div>

    <script>
        const simData = {json_data};
        let currentNet = null, currentExp = null, currentRound = -1;
        let playing = false, playInterval = null, chartHistory = [];
        let lockedNode = null;

        const cFree = "#4b4b4b", cAdv = "#ff4757", cBlock = "#1e90ff", cTrans = "#ffa502";

        const svg = d3.select("#graph svg");
        const container = svg.append("g");

        svg.on("click", (event) => {{
            if (event.target.tagName === 'svg') {{
                lockedNode = null;
                resetHighlights();
            }}
        }});

        svg.call(d3.zoom().scaleExtent([0.1, 5]).on("zoom", e => container.attr("transform", e.transform)));

        const chartSvg = d3.select("#lineChart").append("g").attr("transform", "translate(35,10)");
        const xChart = d3.scaleLinear().domain([1, 30]).range([0, 320]);
        const yChart = d3.scaleLinear().domain([0, 1]).range([60, 0]);
        chartSvg.append("g").attr("class", "axis").attr("transform", "translate(0,60)").call(d3.axisBottom(xChart).ticks(5));
        chartSvg.append("g").attr("class", "axis").call(d3.axisLeft(yChart).ticks(3));
        const lineGen = d3.line().x(d => xChart(d.round)).y(d => yChart(d.val)).curve(d3.curveMonotoneX);
        const chartPath = chartSvg.append("path").attr("class", "line");

        let linkElements, nodeElements, adjList = {{}};
        const labelContainer = container.append("g").attr("class", "labels");

        const netSelect = document.getElementById('netSelect');
        const expSelect = document.getElementById('expSelect');
        simData.forEach((d, i) => {{
            let opt = document.createElement('option');
            opt.value = i; opt.innerHTML = d.network;
            netSelect.appendChild(opt);
        }});

        function populateExperiments(netIndex) {{
            expSelect.innerHTML = '';
            Object.keys(simData[netIndex].experiments).forEach(exp => {{
                let opt = document.createElement('option');
                opt.value = exp; opt.innerHTML = exp;
                expSelect.appendChild(opt);
            }});
        }}

        netSelect.addEventListener('change', e => {{ populateExperiments(e.target.value); loadGame(e.target.value, expSelect.value); }});
        expSelect.addEventListener('change', e => loadGame(netSelect.value, e.target.value));

        function loadGame(netIndex, expName) {{
            currentNet = simData[netIndex]; currentExp = currentNet.experiments[expName];
            currentRound = -1; chartHistory = []; lockedNode = null;

            adjList = {{}};
            currentNet.nodes.forEach(n => adjList[n.id] = []);
            currentNet.links.forEach(l => {{ adjList[l.source].push(l.target); adjList[l.target].push(l.source); }});

            labelContainer.selectAll("*").remove();
            container.selectAll("line").remove();
            container.selectAll("circle").remove();

            const width = document.getElementById('graph').clientWidth;
            const height = document.getElementById('graph').clientHeight;
            const xScale = d3.scaleLinear().domain(d3.extent(currentNet.nodes, d => d.x)).range([80, width-80]);
            const yScale = d3.scaleLinear().domain(d3.extent(currentNet.nodes, d => d.y)).range([80, height-80]);

            currentNet.nodes.forEach(d => {{ d.sx = xScale(d.x); d.sy = yScale(d.y); }});

            linkElements = container.append("g").selectAll("line").data(currentNet.links).enter().append("line")
                .attr("class", "link").attr("x1", d => d.source.sx || currentNet.nodes[d.source].sx).attr("y1", d => d.source.sy || currentNet.nodes[d.source].sy)
                .attr("x2", d => d.target.sx || currentNet.nodes[d.target].sx).attr("y2", d => d.target.sy || currentNet.nodes[d.target].sy);

            nodeElements = container.append("g").selectAll("circle").data(currentNet.nodes).enter().append("circle")
                .attr("class", "node").attr("r", d => Math.max(3.5, Math.sqrt(d.degree) * 2.5))
                .attr("cx", d => d.sx).attr("cy", d => d.sy).attr("fill", cFree)
                .on("mouseover", (event, d) => {{
                    if (!lockedNode) highlightNode(d.id);

                    let status = "Free";
                    let actionInfo = "";
                    if(currentRound >= 0) {{
                        const rd = currentExp[currentRound];

                        // Extract historical strat if present
                        if(rd.adv_history[d.id]) {{
                            actionInfo = `<div style='color:var(--adv-color); font-weight:bold; margin-top:8px; padding-top:6px; border-top:1px solid rgba(255,255,255,0.2)'>⚔️ Adv Strategy:<br>${{rd.adv_history[d.id]}}</div>`;
                        }} else if(rd.blk_history[d.id]) {{
                            actionInfo = `<div style='color:var(--blk-color); font-weight:bold; margin-top:8px; padding-top:6px; border-top:1px solid rgba(255,255,255,0.2)'>🛡️ Blocker Strategy:<br>${{rd.blk_history[d.id]}}</div>`;
                        }}

                        if(rd.adv_seeds.includes(d.id)) status = `<span style='color:var(--adv-color)'>Adversary Seed</span>`;
                        else if(rd.block_seeds.includes(d.id)) status = `<span style='color:var(--blk-color)'>Immunized Node</span>`;
                        else if(rd.transient_nodes.includes(d.id)) status = `<span style='color:var(--trans-color)'>Transient Spread</span>`;
                    }}

                    d3.select("#tooltip").style("opacity", 1).html(`
                        <div class="tooltip-title">Node ID: ${{d.id}}</div>
                        <div style="margin-bottom:4px;">Degree (Hub Size): <b>${{d.degree}}</b></div>
                        <div>Status: <b>${{status}}</b></div>
                        ${{actionInfo}}
                    `).style("left", (event.pageX+15)+"px").style("top", (event.pageY-20)+"px");
                }})
                .on("mouseout", () => {{
                    if (!lockedNode) resetHighlights();
                    d3.select("#tooltip").style("opacity", 0);
                }})
                .on("click", (event, d) => {{
                    event.stopPropagation();
                    if (lockedNode === d.id) {{
                        lockedNode = null;
                        resetHighlights();
                    }} else {{
                        lockedNode = d.id;
                        highlightNode(d.id);
                    }}
                }});

            resetUI();
        }}

        function highlightNode(nodeId) {{
            const neighbors = adjList[nodeId];
            nodeElements.style("opacity", n => (n.id === nodeId || neighbors.includes(n.id)) ? 1 : 0.15);
            linkElements.style("stroke-opacity", l => (l.source === nodeId || l.target === nodeId) ? 0.8 : 0.05)
                        .style("stroke", l => (l.source === nodeId || l.target === nodeId) ? cTrans : "#57606f")
                        .style("stroke-width", l => (l.source === nodeId || l.target === nodeId) ? 2 : 1);
        }}

        function resetHighlights() {{
            nodeElements.style("opacity", 1);
            linkElements.style("stroke-opacity", 0.3).style("stroke", "#57606f").style("stroke-width", 1);
        }}

        function addLabel(nodeId, text, color, yOffset) {{
            if (nodeId == null) return;
            const n = currentNet.nodes.find(d => d.id === nodeId);
            if (!n) return;

            const g = labelContainer.append("g")
                .attr("transform", `translate(${{n.sx}}, ${{n.sy + yOffset}})`)
                .style("pointer-events", "none")
                .style("filter", "drop-shadow(0px 5px 8px rgba(0,0,0,0.85))");

            let fontSize = "14px";
            let fontWeight = "800";

            const textElement = g.append("text")
                .text(text)
                .attr("text-anchor", "middle")
                .attr("font-size", fontSize)
                .attr("font-weight", fontWeight)
                .attr("fill", "#ffffff")
                .attr("dy", "0.35em");

            const bbox = textElement.node().getBBox();

            let paddingX = 12;
            let paddingY = 6;
            let bgAlpha = 0.95;

            g.insert("rect", "text")
                .attr("x", bbox.x - paddingX)
                .attr("y", bbox.y - paddingY)
                .attr("width", bbox.width + paddingX*2)
                .attr("height", bbox.height + paddingY*2)
                .attr("rx", 6)
                .attr("fill", `rgba(18, 18, 18, ${{bgAlpha}})`)
                .attr("stroke", color)
                .attr("stroke-width", 2);
        }}

        function updateRound() {{
            if (currentRound >= currentExp.length) {{ pauseSim(); currentRound--; return; }}
            const rd = currentExp[currentRound];

            document.getElementById('lblRound').innerText = rd.round;
            document.getElementById('lblAdvStrat').innerText = rd.adv_strategy;
            document.getElementById('lblBlockStrat').innerText = rd.block_strategy;
            document.getElementById('lblAbsInf').innerText = rd.abs_influence;

            let infPct = (rd.norm_influence * 100).toFixed(1);
            document.getElementById('infBar').style.width = infPct + "%";
            document.getElementById('infText').innerText = infPct + "%";

            chartHistory.push({{round: rd.round, val: rd.norm_influence}});
            chartPath.datum(chartHistory).transition().duration(300).attr("d", lineGen);

            nodeElements.classed("node-new-adv", false).classed("node-new-blk", false);
            labelContainer.selectAll("*").remove();

            nodeElements.classed("node-new-adv", d => d.id === rd.new_adv);
            nodeElements.classed("node-new-blk", d => d.id === rd.new_blk);

            // Only draw pills for the currently selected nodes in this round
            if (rd.new_adv !== null && rd.new_adv !== undefined) {{
                addLabel(rd.new_adv, "Adv: " + rd.adv_strategy, cAdv, -35);
            }}
            if (rd.new_blk !== null && rd.new_blk !== undefined) {{
                addLabel(rd.new_blk, "Blk: " + rd.block_strategy, cBlock, 35);
            }}

            nodeElements.filter(d => d.id === rd.new_adv || d.id === rd.new_blk).raise();

            nodeElements.interrupt();

            nodeElements.transition("wipe")
                .duration(400)
                .attr("fill", d => {{
                    if (rd.adv_seeds.includes(d.id)) return cAdv;
                    if (rd.block_seeds.includes(d.id)) return cBlock;
                    return cFree;
                }})
                .attr("stroke", "#121212")
                .attr("stroke-width", 1)
              .transition("bloom")
                .delay(1000)
                .duration(800)
                .attr("fill", d => {{
                    if (rd.transient_nodes.includes(d.id)) return cTrans;
                    if (rd.adv_seeds.includes(d.id)) return cAdv;
                    if (rd.block_seeds.includes(d.id)) return cBlock;
                    return cFree;
                }});

            // Ensure labels stay on top of the graph after element animation
            labelContainer.raise();
        }}

        function resetUI() {{
            currentRound = -1; chartHistory = []; chartPath.datum(chartHistory).attr("d", lineGen); lockedNode = null; resetHighlights();
            labelContainer.selectAll("*").remove();
            document.getElementById('lblRound').innerText = '0';
            document.getElementById('lblAdvStrat').innerText = '-';
            document.getElementById('lblBlockStrat').innerText = '-';
            document.getElementById('lblAbsInf').innerText = '0';
            document.getElementById('infBar').style.width = "0%";
            document.getElementById('infText').innerText = "0.0%";
            nodeElements.classed("node-new-adv", false).classed("node-new-blk", false);
            nodeElements.interrupt().transition().duration(200).attr("fill", cFree).attr("stroke", "#121212").attr("stroke-width", 1);
        }}

        function playSim() {{
            if (currentRound >= currentExp.length - 1) resetUI();
            document.getElementById('btnPlay').disabled = true; document.getElementById('btnPause').disabled = false; document.getElementById('btnNext').disabled = true;
            playInterval = setInterval(() => {{ currentRound++; updateRound(); }}, 2800);
        }}

        function pauseSim() {{
            clearInterval(playInterval);
            document.getElementById('btnPlay').disabled = false; document.getElementById('btnPause').disabled = true; document.getElementById('btnNext').disabled = false;
        }}

        document.getElementById('btnPlay').addEventListener('click', playSim);
        document.getElementById('btnPause').addEventListener('click', pauseSim);
        document.getElementById('btnNext').addEventListener('click', () => {{ currentRound++; updateRound(); }});
        document.getElementById('btnReset').addEventListener('click', () => {{ pauseSim(); resetUI(); }});

        labelContainer.raise();

        // Hide loader once graphs are ready
        setTimeout(() => {{
            if(simData.length > 0) {{ populateExperiments(0); loadGame(0, expSelect.value); }}
            const loader = document.getElementById('loading-overlay');
            loader.style.opacity = '0';
            setTimeout(() => loader.style.visibility = 'hidden', 600);
        }}, 500);

    </script>
</body>
</html>"""

with open('interactive_simulation_dark.html', 'w', encoding='utf-8') as f:
    f.write(html_content)

print("Dark Mode Strategy Simulation Generated successfully!")
files.download('interactive_simulation_dark.html')

Preparing graph structures and computing metrics...


Processing Networks: 100%|██████████| 3/3 [00:13<00:00,  4.64s/it]

Dark Mode Strategy Simulation Generated successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>